> **日本語版について**: このノートブックは、[jpmorganchase/python-training](https://github.com/jpmorganchase/python-training) の英語原文をもとにした日本語版です（Apache License 2.0）。原文にはタイトルも説明文もなく、コードだけが置かれています。以下の「補足」は日本語版で独自に追加した解説で、原文の翻訳ではありません。コード自体は原文から変更していません。

# 3次元プロット

**補足（日本語版独自の解説）: このノートブックが描くもの**

描かれるのは、米国債のイールドカーブ（期間ごとの利回りを並べた曲線）を、1990年から2015年まで毎日分重ねた3次元の「面」です。

- 横軸: 期間（1か月〜30年の11種類）
- 奥行き: 日付（1990年1月2日〜2015年3月18日、6,307営業日）
- 高さ: 利回り（%）

イールドカーブは通常、右上がりの曲線です。それを25年分並べると面になり、金利環境の変化が一望できます。面全体が時代とともに低くなっていく様子が、長期の金利低下そのものです。短期側（手前）が長期側より高くなる箇所は「逆イールド」で、景気後退の前触れとして知られます。そういう時期が面の形に現れていることも読み取れます。

データは `../data/yield_curve.csv` です。列の構成は次のとおりで、描画ライブラリに渡す形に最初から整えられています。

- `x` 列: 期間のラベル（先頭の11行にだけ入っている）
- `y` 列: 日付
- `z[0]`〜`z[10]` 列: その日の各期間の利回り

次のセルは、`x` と `y` を軸用に取り出し、残りを高さの行列 `zlist` に変える処理です。

In [ ]:
import pandas as pd
import numpy as np

import plotly.graph_objs as go

df = pd.read_csv('../data/yield_curve.csv')
xlist = list(df["x"].dropna())
ylist = list(df["y"].dropna())

del df["x"]
del df["y"]

zlist = []
for row in df.iterrows():
    index, data = row
    zlist.append(data.tolist())


**補足（日本語版独自の解説）: 濃い線と `"None"` の処理**

上のセルで描いているのは2つです。

- `trace1`: 利回りの面そのもの（Surface）
- `trace2`: 面の手前の縁に沿う濃い線。最も短い期間の利回りの推移だけをなぞっています

線をわざわざ継ぎはぎしているのには理由があります。最短の1か月物にあたる列 `z[0]` は、1990年1月2日から2001年7月30日までの2,899行が文字列 `"None"`、つまり値がありません。当時はその期間の系列自体が存在しなかったためです。そこでコードは、値のない期間は次の列 `z[1]`（3か月物）で代用し、2001年7月31日以降は `z[0]`（1か月物）に切り替えて、1本の線につないでいます。

中・長期側の列にも欠けた期間があります（`z[9]` で939行、`z[10]` で994行が `"None"`）。文字列は数値として扱えないので、その箇所は面に穴が開いた状態で描かれます。

なお `lighting` や `colorscale` は見た目の調整です。意味を持つのは、軸と値の対応だけです。

In [ ]:
z_secondary_beginning = [z[1] for z in zlist if z[0] == "None"]
z_secondary_end = [z[0] for z in zlist if z[0] != "None"]
z_secondary = z_secondary_beginning + z_secondary_end
x_secondary = ["3-month"] * len(z_secondary_beginning) + ["1-month"] * len(
    z_secondary_end
)
y_secondary = ylist

trace1 = dict(
    type="surface",
    x=xlist,
    y=ylist,
    z=zlist,
    hoverinfo="x+y+z",
    lighting={
        "ambient": 0.95,
        "diffuse": 0.99,
        "fresnel": 0.01,
        "roughness": 0.01,
        "specular": 0.01,
    },
    colorscale=[
        [0, "rgb(230,245,254)"],
        [0.4, "rgb(123,171,203)"],
        [0.8, "rgb(40,119,174)"],
        [1, "rgb(37,61,81)"],
    ],
    opacity=0.7,
    showscale=False,
    scene="scene",
)

trace2 = dict(
    type="scatter3d",
    mode="lines",
    x=x_secondary,
    y=y_secondary,
    z=z_secondary,
    hoverinfo="x+y+z",
    line=dict(color="#444444"),
)

data = [trace1, trace2]
layout = dict(
    autosize=True,
    font=dict(size=12, color="#CCCCCC"),
    margin=dict(t=5, l=5, b=5, r=5),
    showlegend=False,
    hovermode="closest",
    scene=dict(
        aspectmode="manual",
        aspectratio=dict(x=2, y=5, z=1.5),
        xaxis={
            "showgrid": True,
            "title": "",
            "type": "category",
            "zeroline": False,
            "categoryorder": "array",
            "categoryarray": list(reversed(xlist)),
        },
        yaxis={"showgrid": True, "title": "", "type": "date", "zeroline": False},
    ),
)
figure = dict(data=data, layout=layout)

go.FigureWidget(figure)